# TweetAge · 01 Exploración de los datos

Análisis descriptivo del corpus de entrenamiento (y, cuando aplica, del de test) **por rango de edad**. Es la sección de *análisis de datos* del entregable.

Todo lo de este notebook es **descriptivo**: tablas y gráficos para entender el corpus y justificar decisiones. Nada de lo que se calcula aquí alimenta a los modelos (que siguen siendo los LMs de n-gramas de `tweetage.models`).

**Estructura**

1. Volumen, balance y calidad del texto.
2. Duplicados, plantillas y solapamiento train/test.
3. Longitud.
4. Rasgos de estilo, menciones, hashtags y URLs.
5. Vocabulario y n-gramas característicos de cada clase.
6. Conclusiones (y observaciones para revisar después).

In [ ]:
import platform
import re
import sys
import unicodedata
from collections import Counter

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import nltk
import sklearn
from nltk.corpus import stopwords

from tweetage import paths, validation
from tweetage import pipeline as P
from tweetage.models import TOKEN_RE

CLASES = P.CLASES
SEED = 20261108  # solo se usa para muestrear ejemplos
FIG_DIR = paths.work_dir() / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 150,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
    }
)
pd.set_option("display.max_colwidth", 90)

print(f"Python {sys.version.split()[0]} | {platform.system()} | entorno: {paths.environment()}")
print(
    f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__} | matplotlib {matplotlib.__version__}"
)

In [ ]:
def por_clase(df, cols, como_pct=True):
    # Media de cada columna por clase + total; en % si las columnas son booleanas.
    tabla = df.groupby("clase")[cols].mean().reindex(CLASES).T
    tabla["total"] = df[cols].mean()
    return (tabla * 100).round(1) if como_pct else tabla.round(1)


def tasa(textos, patron, flags=0):
    # Indicador por tweet: ¿el patrón aparece al menos una vez?
    rx = re.compile(patron, flags)
    return np.fromiter((rx.search(t) is not None for t in textos), dtype=bool, count=len(textos))


def guardar(fig, nombre):
    fig.savefig(FIG_DIR / f"01_{nombre}.png", bbox_inches="tight")

## 1. Volumen, balance y calidad del texto

Se usan los textos tal como los ven los modelos (`paths.texts`: nulos → cadena vacía y sin espacios en los extremos; los CSV traen relleno de espacios a la derecha).

In [ ]:
train, test = paths.load_data()
X = paths.texts(train)
X_test = paths.texts(test)
y = train["age_range"].to_numpy()

balance = train["age_range"].value_counts().reindex(CLASES).to_frame("tweets")
balance["%"] = (100 * balance["tweets"] / balance["tweets"].sum()).round(2)
print(f"Train: {len(X):,} tweets | test: {len(X_test):,} tweets")
print(f"Textos vacíos: train {sum(t == '' for t in X)} | test {sum(t == '' for t in X_test)}")
print(f"Longitud máxima: {max(map(len, X))} caracteres")
balance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
barras = ax.bar(CLASES, balance["tweets"], color="#4C72B0")
ax.bar_label(
    barras,
    labels=[f"{n:,}\n{p:.1f} %" for n, p in zip(balance["tweets"], balance["%"])],
    padding=2,
    fontsize=8,
)
ax.set_ylim(0, balance["tweets"].max() * 1.18)  # espacio para las etiquetas
ax.set_ylabel("tweets")
ax.set_title("Tweets por rango de edad (train)")
guardar(fig, "balance")
plt.show()

**Codificación del texto.** Antes de medir estilo hay que saber qué caracteres existen en el corpus.

In [ ]:
no_ascii_train = Counter(ch for t in X for ch in t if ord(ch) > 127)
no_ascii_test = Counter(ch for t in X_test for ch in t if ord(ch) > 127)
print("Caracteres no ASCII en train:", dict(no_ascii_train))
print("Caracteres no ASCII en test: ", dict(no_ascii_test))
print(
    "Tweets con 'ñ', tildes, '¡' o emojis en train:",
    sum(bool(re.search(r"[ñÑáéíóúÁÉÍÓÚ¡\U0001F300-\U0001FAFF☀-➿]", t)) for t in X),
)

artefactos = pd.DataFrame(
    {
        "clase": y,
        "#¿NOMBRE? (error de Excel)": tasa(X, r"#¿NOMBRE\?"),
        "entidad HTML (&lt; &gt; &amp;)": tasa(X, r"&(?:lt|gt|amp|quot);"),
        "llega a 140 caracteres": np.array([len(t) >= 140 for t in X]),
    }
)
cols = list(artefactos.columns[1:])
print("\nConteo total:", artefactos[cols].sum().to_dict())
por_clase(artefactos, cols)

In [ ]:
# Ejemplos de palabras que en español llevan tilde o ñ, tal como aparecen en el corpus
muestra = [t for t in X if re.search(r"\b(?:ano|companeros|nino|manana|espana|accion|corazon)\b", t, re.I)]
pd.Series(muestra).sample(6, random_state=SEED).to_frame("texto")

## 2. Duplicados, plantillas y solapamiento train/test

Dos niveles de normalización:

- **N1** = la misma que usa `validation.groups_of` para agrupar la partición (`strip` + `casefold`). Detecta copias exactas.
- **N2** = N1 + URLs → `URL`, menciones → `@U`, hashtags → `#H`, números → `0` y letras repetidas colapsadas. Detecta **plantillas**: tweets que solo cambian en el enlace o el usuario (retweets, mensajes de apps, tweets sin texto propio).

In [ ]:
def n2(texto):
    t = re.sub(r"\s+", " ", texto.strip().casefold())
    t = re.sub(r"https?://\S*|www\.\S+", "URL", t)
    t = re.sub(r"@\w+", "@U", t)
    t = re.sub(r"#\w+", "#H", t)
    t = re.sub(r"\d+", "0", t)
    return re.sub(r"(.)\1{2,}", r"\1\1", t)


def resumen_duplicados(grupos_train, grupos_test, nivel):
    df = pd.DataFrame({"g": grupos_train, "clase": y})
    g = df.groupby("g")["clase"].agg(n="size", etiquetas="nunique")
    rep = g[g["n"] > 1]
    en_train = pd.Series(grupos_test).isin(set(grupos_train))
    return {
        "nivel": nivel,
        "textos distintos": len(g),
        "tweets que se repiten": int(rep["n"].sum()),
        "% del train": round(100 * rep["n"].sum() / len(df), 2),
        "textos repetidos con >1 etiqueta": int((rep["etiquetas"] > 1).sum()),
        "tweets con etiqueta ambigua": int(rep.loc[rep["etiquetas"] > 1, "n"].sum()),
        "tweets de test vistos en train": int(en_train.sum()),
        "% del test": round(100 * en_train.mean(), 2),
    }, g


g1_train, g1_test = validation.groups_of(X), validation.groups_of(X_test)
g2_train, g2_test = [n2(t) for t in X], [n2(t) for t in X_test]
r1, grupos_n1 = resumen_duplicados(g1_train, g1_test, "N1 (copia exacta)")
r2, grupos_n2 = resumen_duplicados(g2_train, g2_test, "N2 (plantilla)")
pd.DataFrame([r1, r2]).set_index("nivel").T

In [ ]:
# Copias exactas más frecuentes y cómo se reparten entre clases
def reparto(grupos, claves):
    df = pd.DataFrame({"g": grupos, "clase": y})
    df = df[df["g"].isin(claves)]
    return pd.crosstab(df["g"], df["clase"]).reindex(columns=CLASES, fill_value=0).reindex(claves)


top_n1 = grupos_n1.sort_values("n", ascending=False).head(12).index.tolist()
reparto(g1_train, top_n1).rename(index={"": "(texto vacío)"})

In [ ]:
# Plantillas más frecuentes (N2): tweets cuyo único contenido es un enlace, un RT o un mensaje automático
top_n2 = grupos_n2.sort_values("n", ascending=False).head(15).index.tolist()
tabla_n2 = reparto(g2_train, top_n2)
tabla_n2["total"] = tabla_n2.sum(axis=1)
tabla_n2

In [ ]:
# Tweets sin texto propio: al quitar URLs, menciones, hashtags, "RT" y puntuación no queda ninguna palabra
sin_texto = np.array(
    [
        re.search(r"[a-z]{2,}", re.sub(r"https?://\S*|www\.\S+|@\w+|#\w+|\brt\b", " ", t.casefold())) is None
        for t in X
    ]
)
print(f"Tweets sin texto propio: {sin_texto.sum():,} ({sin_texto.mean():.1%})")
por_clase(pd.DataFrame({"clase": y, "sin texto propio": sin_texto}), ["sin texto propio"])

### Muestras de las plantillas

Tweets reales detrás de las plantillas: las más frecuentes, las que mezclan etiquetas (ruido) y las casi exclusivas de una clase (≥ 90 %, casi siempre bots o una misma cuenta).

In [ ]:
# Muestras de tweets originales detrás de cada plantilla N2 (las más frecuentes)
muestras_n2 = pd.DataFrame({"plantilla": g2_train, "texto": X, "clase": y})


def ver_plantilla(plantilla, n=5):
    sub = muestras_n2[muestras_n2["plantilla"] == plantilla]
    return sub.sample(min(n, len(sub)), random_state=SEED)[["clase", "texto"]]


for p in top_n2[:8]:
    print(f"\n=== {p!r}  ({(muestras_n2['plantilla'] == p).sum()} tweets)")
    display(ver_plantilla(p))

In [ ]:
# Plantillas con etiquetas mezcladas (ruido): mismo "esqueleto", distintas clases
mixtas = grupos_n2[(grupos_n2["n"] >= 5) & (grupos_n2["etiquetas"] > 1)].sort_values("n", ascending=False)
print(f"Plantillas (≥5 tweets) con más de una etiqueta: {len(mixtas):,}")
p = mixtas.index[10]  # cambia el índice para ver otra
print(f"Ejemplo: {p!r}")
ver_plantilla(p, n=8)

In [ ]:
# Plantillas casi exclusivas de una clase (≥20 tweets y ≥90 % en la misma clase)
pureza_n2 = muestras_n2.groupby("plantilla")["clase"].agg(
    n="size",
    clase_principal=lambda s: s.mode().iloc[0],
    pureza=lambda s: s.value_counts(normalize=True).iloc[0],
)
puras = pureza_n2[(pureza_n2["n"] >= 20) & (pureza_n2["pureza"] >= 0.9)].sort_values("n", ascending=False)
print(f"Plantillas puras: {len(puras)} | tweets que cubren: {puras['n'].sum():,}")
display(puras.head(15).round(2))
for p in puras.index[:3]:
    print(f"\n=== {p!r}")
    display(ver_plantilla(p, n=3))

In [ ]:
# Tweets de test cuya plantilla ya aparece en train, con la distribución de clases de esa plantilla en train
en_train = pd.DataFrame({"plantilla": g2_test, "texto_test": X_test})
en_train = en_train[en_train["plantilla"].isin(pureza_n2.index)].join(pureza_n2, on="plantilla")
print(
    f"Tweets de test con plantilla vista en train: {len(en_train):,} | "
    f"de ellos con plantilla pura (≥90 %, n≥5): {((en_train['pureza'] >= 0.9) & (en_train['n'] >= 5)).sum():,}"
)
en_train.sample(10, random_state=SEED)[["texto_test", "plantilla", "n", "clase_principal", "pureza"]].round(2)

## 3. Longitud

Longitud en caracteres y en tokens (mismo tokenizador de los modelos: URLs, menciones, hashtags, palabras y signos sueltos). Twitter cortaba en 140 caracteres, así que hay un techo duro.

In [ ]:
longitud = pd.DataFrame(
    {
        "clase": y,
        "caracteres": [len(t) for t in X],
        "tokens": [len(TOKEN_RE.findall(t)) for t in X],
    }
)
resumen_long = (
    longitud.groupby("clase")[["caracteres", "tokens"]]
    .agg(["mean", "median", lambda s: s.quantile(0.1), lambda s: s.quantile(0.9)])
    .reindex(CLASES)
    .round(1)
)
resumen_long.columns = [
    f"{a}_{b}" for a, b in zip(np.repeat(["caracteres", "tokens"], 4), ["media", "mediana", "p10", "p90"] * 2)
]
resumen_long

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, col in zip(axes, ["caracteres", "tokens"]):
    datos = [longitud.loc[longitud["clase"] == c, col] for c in CLASES]
    ax.boxplot(datos, showfliers=False, medianprops={"color": "#C44E52", "lw": 2})
    ax.set_xticks(range(1, 7), CLASES)
    ax.plot(range(1, 7), [d.mean() for d in datos], "o", color="#4C72B0", ms=5, label="media")
    ax.set_title(f"Longitud en {col}")
    ax.set_xlabel("rango de edad")
axes[0].legend(frameon=False, loc="lower right")
guardar(fig, "longitud")
plt.show()

In [ ]:
# Distribución completa de la longitud en caracteres: dónde se concentran los tweets de cada clase
fig, ax = plt.subplots(figsize=(7, 3.4))
bins = np.arange(0, 145, 5)
for c, color in zip(CLASES, plt.cm.viridis(np.linspace(0, 0.9, 6))):
    ax.hist(
        longitud.loc[longitud["clase"] == c, "caracteres"],
        bins=bins,
        density=True,
        histtype="step",
        lw=1.6,
        color=color,
        label=c,
    )
ax.set_xlabel("caracteres")
ax.set_ylabel("densidad")
ax.set_title("Distribución de la longitud por rango de edad")
ax.legend(frameon=False, ncol=2)
guardar(fig, "longitud_hist")
plt.show()

## 4. Rasgos de estilo, menciones, hashtags y URLs

Cada rasgo es un indicador por tweet (¿aparece al menos una vez?) y la tabla muestra el **% de tweets de cada clase** que lo tienen. Se calcula sobre el texto original, **antes** del `casefold` que aplican los modelos.

| Rasgo | Definición |
|---|---|
| RT | empieza con `RT @usuario` |
| mención al inicio | el primer token es `@usuario` (respuesta) |
| mención dentro | hay `@usuario` pero no al inicio |
| mayúscula inicial | la primera letra del texto propio (sin `RT @x:` ni menciones iniciales) es mayúscula |
| gritado | ≥ 70 % de las letras en mayúscula y al menos 10 letras |
| alargamiento | una letra repetida 3+ veces (`siii`, `nooo`) |
| risa | `jaja`, `jeje`, `jsjs`, `xd`, `haha`, `lol` |
| emoticono | `:)`, `:(`, `:D`, `;)`, `:P`, `<3` |
| abreviatura | `q`, `xq`, `pq`, `tb`, `tmb`, `xfa`, `porfa`, `bn`, `dnd`, `k`, `ke`, `d` |
| espacio doble | dos o más espacios seguidos dentro del texto |

In [ ]:
texto_propio = [re.sub(r"^(?:RT @\w+:?\s*)?(?:@\w+\s*)*", "", t) for t in X]
letras = np.array([sum(ch.isalpha() for ch in t) for t in X])
mayus = np.array([sum(ch.isupper() for ch in t) for t in X])
pct_mayus = np.divide(mayus, letras, out=np.zeros(len(X)), where=letras > 0)
primera = [re.search(r"[A-Za-z]", t) for t in texto_propio]

rasgos = pd.DataFrame(
    {
        "clase": y,
        # forma
        "mayúscula inicial": [m is not None and m.group().isupper() for m in primera],
        "gritado": (pct_mayus >= 0.7) & (letras >= 10),
        # puntuación
        "pregunta ?": tasa(X, r"\?"),
        "exclamación !": tasa(X, r"!"),
        "!! o más": tasa(X, r"!{2,}"),
        "puntos suspensivos": tasa(X, r"\.{3,}"),
        "espacio doble": tasa(X, r"\S {2,}\S"),
        # expresividad
        "alargamiento": tasa(X, r"([a-z])\1{2,}", re.I),
        "risa": tasa(X, r"\b(?:j[aeiou]){2,}j?\b|\bjs(?:js)+\b|\bx+d+\b|\b(?:ha){2,}\b|\blol\b", re.I),
        "emoticono": tasa(X, r"(?<!\w)[:;=][-']?[)(DPpOo3](?!\w)|<3|&lt;3"),
        "abreviatura": tasa(X, r"\b(?:q|xq|pq|tb|tmb|xfa|porfa|bn|dnd|k|ke|d)\b", re.I),
        "número": tasa(X, r"\d"),
        # interacción y enlaces
        "RT": tasa(X, r"^RT @\w+"),
        "mención al inicio": tasa(X, r"^@\w+"),
        "mención dentro": tasa(X, r"(?<!^)(?<=\W)@\w+") & ~tasa(X, r"^@\w+"),
        "hashtag": tasa(X, r"#\w+"),
        "URL": tasa(X, r"https?://|www\."),
    }
)
cols_rasgos = list(rasgos.columns[1:])
tabla_rasgos = por_clase(rasgos, cols_rasgos)
tabla_rasgos.loc["% de letras en mayúscula"] = pd.Series(pct_mayus).groupby(y).mean().reindex(CLASES).mul(
    100
).round(1).tolist() + [round(100 * pct_mayus.mean(), 1)]
tabla_rasgos

In [ ]:
# Cada celda = tasa de la clase / tasa global. >1 (rojo) la clase lo usa más que el promedio; <1 (azul), menos.
razon = tabla_rasgos[CLASES].div(tabla_rasgos["total"], axis=0)
fig, ax = plt.subplots(figsize=(7.5, 6.8))
im = ax.imshow(np.log2(razon.to_numpy()), cmap="RdBu_r", vmin=-1.5, vmax=1.5, aspect="auto")
ax.set_xticks(range(6), CLASES)
ax.set_yticks(range(len(razon)), razon.index)
for i in range(razon.shape[0]):
    for j in range(razon.shape[1]):
        ax.text(j, i, f"{tabla_rasgos.iloc[i, j]:.1f}", ha="center", va="center", fontsize=8)
ax.set_title("Rasgos de estilo por clase\n(número = % de tweets; color = log2 de la razón frente al total)")
fig.colorbar(im, ax=ax, shrink=0.6, label="log2(tasa clase / tasa global)")
guardar(fig, "rasgos_estilo")
plt.show()

In [ ]:
# Foco en las clases más difíciles: diferencia en puntos porcentuales 25-34 menos 18-24
dif = (tabla_rasgos["25-34"] - tabla_rasgos["18-24"]).sort_values()
fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(dif.index, dif.values, color=np.where(dif.values > 0, "#C44E52", "#4C72B0"))
ax.axvline(0, color="gray", lw=1)
ax.set_xlabel("puntos porcentuales (25-34 − 18-24)")
ax.set_title("¿Qué separa 18-24 de 25-34?")
guardar(fig, "rasgos_18_24_vs_25_34")
plt.show()

### Menciones: ¿a quién se menciona?

Las menciones fueron el canal que más aportó a la mezcla. Aquí se mide qué tan **exclusivo de una clase** es cada usuario mencionado (pureza = % de sus apariciones en la clase más frecuente) y qué fracción de las menciones de test ya aparece en train.

In [ ]:
def handles(texto):
    return {h.casefold() for h in re.findall(r"@\w+", texto)}


menciones = pd.DataFrame([(h, c) for t, c in zip(X, y) for h in handles(t)], columns=["usuario", "clase"])
por_usuario = menciones.groupby("usuario")["clase"].agg(
    n="size",
    pureza=lambda s: s.value_counts(normalize=True).iloc[0],
    clase_principal=lambda s: s.mode().iloc[0],
)
frecuentes = por_usuario[por_usuario["n"] >= 5]

vistos = set(por_usuario.index)
menciones_test = [h for t in X_test for h in handles(t)]
print(f"Usuarios distintos mencionados en train: {len(por_usuario):,}")
print(
    f"Usuarios con ≥5 menciones: {len(frecuentes):,} | pureza media {frecuentes['pureza'].mean():.1%} | "
    f"con pureza ≥90 %: {(frecuentes['pureza'] >= 0.9).mean():.1%}"
)
print(
    f"Menciones de test cuyo usuario ya aparece en train: {np.mean([h in vistos for h in menciones_test]):.1%}"
)
por_usuario.sort_values("n", ascending=False).head(12).round(3)

In [ ]:
# Usuarios más mencionados y casi exclusivos de una clase (pureza ≥ 95 %), por clase
(
    frecuentes[frecuentes["pureza"] >= 0.95]
    .sort_values("n", ascending=False)
    .groupby("clase_principal")
    .head(3)
    .sort_values(["clase_principal", "n"], ascending=[True, False])
    .round(3)
)

## 5. Vocabulario característico de cada clase

Se usa el **log-odds con prior de Dirichlet informativo** (Monroe, Colaresi y Quinn, 2008): compara la frecuencia de una palabra en una clase contra el resto, suavizada con su frecuencia global, y la divide por su desviación estándar (*z-score*). A diferencia de la frecuencia cruda, no premia palabras comunes en todas las clases ni palabras raras con pocos casos.

Solo palabras: se excluyen URLs, menciones, hashtags, números, signos y tokens de una letra; mínimo 20 apariciones en total. Prior: 1 000 pseudo-conteos repartidos según la frecuencia global.

In [ ]:
def palabras(texto):
    return [t for t in TOKEN_RE.findall(texto.casefold()) if re.fullmatch(r"[a-z_]{2,}", t) and t != "rt"]


conteos = {c: Counter() for c in CLASES}
for t, c in zip(X, y):
    conteos[c].update(palabras(t))
total = sum(conteos.values(), Counter())
vocab = [w for w, n in total.items() if n >= 20]
print(f"Vocabulario analizado: {len(vocab):,} palabras")


def log_odds(cnt_a, cnt_b, prior_total=1000):
    # z-score del log-odds de cada palabra en a frente a b, con prior de Dirichlet informativo
    n_glob = sum(total[w] for w in vocab)
    a = np.array([prior_total * total[w] / n_glob for w in vocab])
    ya = np.array([cnt_a[w] for w in vocab], dtype=float)
    yb = np.array([cnt_b[w] for w in vocab], dtype=float)
    na, nb, a0 = ya.sum(), yb.sum(), a.sum()
    delta = np.log((ya + a) / (na + a0 - ya - a)) - np.log((yb + a) / (nb + a0 - yb - a))
    z = delta / np.sqrt(1 / (ya + a) + 1 / (yb + a))
    return pd.Series(z, index=vocab)


**Palabras más características de cada clase.** Las stopwords (lista de NLTK en español e inglés, sin tildes porque el corpus es ASCII) se quitan **solo de la tabla**: el *z-score* se calcula sobre todo el vocabulario. Entre paréntesis el *z-score* (|z| > 2 ≈ diferencia significativa al 5 %).

In [ ]:
nltk.download("stopwords", quiet=True)  # solo descarga la primera vez


def sin_tilde(w):
    # El corpus viene en ASCII ("año" -> "ano"), así que las stopwords también deben ir sin tildes
    return "".join(ch for ch in unicodedata.normalize("NFD", w) if unicodedata.category(ch) != "Mn")


STOPWORDS = {sin_tilde(w) for w in stopwords.words("spanish") + stopwords.words("english")}
STOPWORDS |= {"lt", "gt", "amp", "quot", "http", "https", "co", "www", "via"}  # restos de HTML y URLs
print(f"Stopwords: {len(STOPWORDS)}")

z_clase = {c: log_odds(conteos[c], total - conteos[c]) for c in CLASES}
TOP = 20
pd.DataFrame(
    {
        c: [f"{w} ({z:.0f})" for w, z in z_clase[c].drop(STOPWORDS, errors="ignore").nlargest(TOP).items()]
        for c in CLASES
    },
    index=range(1, TOP + 1),
)

In [ ]:
# ¿Qué tan frecuentes son esas palabras? % de tweets de cada clase que contienen al menos una de sus top-20
cobertura = {}
for c in CLASES:
    top_c = set(z_clase[c].drop(STOPWORDS, errors="ignore").nlargest(TOP).index)
    tw = [t for t, l in zip(X, y) if l == c]
    cobertura[c] = 100 * np.mean([bool(top_c & set(palabras(t))) for t in tw])
pd.Series(cobertura, name="% tweets con alguna top-20").round(1).to_frame().T

Entre paréntesis el *z-score* (|z| > 2 ≈ diferencia significativa al 5 %). A continuación, la comparación directa de las dos clases centrales, que son las que el modelo más confunde.

In [ ]:
z_pareja = log_odds(conteos["25-34"], conteos["18-24"])
pd.DataFrame(
    {
        "más propias de 18-24": [f"{w} ({-z:.1f})" for w, z in z_pareja.nsmallest(20).items()],
        "más propias de 25-34": [f"{w} ({z:.1f})" for w, z in z_pareja.nlargest(20).items()],
    },
    index=range(1, 21),
)

**Palabras de etapa de vida.** Frecuencia por cada 10 000 palabras de algunos términos que marcan etapas (estudio, pareja, familia, trabajo).

In [ ]:
etapa = [
    "colegio",
    "examen",
    "novio",
    "novia",
    "mama",
    "bebe",
    "universidad",
    "trabajo",
    "oficina",
    "jefe",
    "esposa",
    "hijos",
]
n_clase = {c: sum(conteos[c].values()) for c in CLASES}
tabla_etapa = pd.DataFrame(
    {c: [1e4 * conteos[c][w] / n_clase[c] for w in etapa] for c in CLASES}, index=etapa
).round(1)
tabla_etapa["z 25-34 vs 18-24"] = [round(z_pareja.get(w, np.nan), 1) for w in etapa]
tabla_etapa

In [ ]:
# Cuántas palabras separan de forma significativa cada par de clases (|z| > 3): mide qué tan distintas son léxicamente
pares = pd.DataFrame(0, index=CLASES, columns=CLASES)
for i, a in enumerate(CLASES):
    for b in CLASES[i + 1 :]:
        n = int((log_odds(conteos[a], conteos[b]).abs() > 3).sum())
        pares.loc[a, b] = pares.loc[b, a] = n

fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(pares, cmap="Greens")
ax.set_xticks(range(6), CLASES)
ax.set_yticks(range(6), CLASES)
for i in range(6):
    for j in range(6):
        if i != j:
            ax.text(j, i, pares.iloc[i, j], ha="center", va="center", fontsize=8)
ax.set_title("Palabras con |z| > 3 entre cada par de clases")
guardar(fig, "distancia_lexica")
plt.show()

### N-gramas característicos

Los canales del modelo son n-gramas de caracteres (4, 5 y 6) y LMs secuenciales, así que también se buscan los **4-gramas de caracteres** y **bigramas de palabras** más característicos de cada clase, con el mismo log-odds (mínimo 30 apariciones). Las URLs y menciones se quitan de los 4-gramas para no mostrar nombres de cuentas.

In [ ]:
def char_ngramas(texto, n):
    # Igual que los canales char: texto en minúsculas; URLs y menciones fuera para no mostrar nombres de cuentas
    t = re.sub(r"https?://\S*|www\.\S+|@\w+", " ", texto.casefold())
    t = re.sub(r"\s+", " ", t)
    return [t[i : i + n] for i in range(len(t) - n + 1)]


def bigramas(texto):
    toks = [
        t
        for t in TOKEN_RE.findall(texto.casefold())
        if not t.startswith(("http", "www.", "@", "#")) and not t.isdigit() and t != "rt"
    ]
    return [f"{a} {b}" for a, b in zip(toks, toks[1:])]


def log_odds_conteos(conteos_clase, min_total=30, prior_total=1000):
    # Mismo log-odds con prior de Dirichlet, para cualquier tipo de evento; devuelve {clase: z-scores}
    tot = sum(conteos_clase.values(), Counter())
    voc = [g for g, n in tot.items() if n >= min_total]
    t = np.array([tot[g] for g in voc], dtype=float)
    a = prior_total * t / t.sum()
    out = {}
    for c in CLASES:
        ya = np.array([conteos_clase[c][g] for g in voc], dtype=float)
        yb = t - ya
        na, nb, a0 = ya.sum(), yb.sum(), a.sum()
        delta = np.log((ya + a) / (na + a0 - ya - a)) - np.log((yb + a) / (nb + a0 - yb - a))
        out[c] = pd.Series(delta / np.sqrt(1 / (ya + a) + 1 / (yb + a)), index=voc)
    return out


def tabla_top(z, k=15):
    return pd.DataFrame(
        {c: [f"{g!r} ({v:.0f})" for g, v in z[c].nlargest(k).items()] for c in CLASES}, index=range(1, k + 1)
    )


cnt_c4 = {c: Counter() for c in CLASES}
cnt_bi = {c: Counter() for c in CLASES}
for t, c in zip(X, y):
    cnt_c4[c].update(char_ngramas(t, 4))
    cnt_bi[c].update(bigramas(t))
z_c4, z_bi = log_odds_conteos(cnt_c4), log_odds_conteos(cnt_bi)
print("4-gramas de caracteres más característicos (comillas para ver espacios)")
tabla_top(z_c4)

In [ ]:
print("Bigramas de palabras más característicos")
tabla_top(z_bi)

### Especificidad frente a dispersión por orden

Al subir el orden *n*, un n-grama es más específico de una clase, pero es más probable no haberlo visto en train. Se mide sobre la partición de ajuste (`validation.split`, semilla 20261108):

- **% no visto**: ocurrencias de validación cuyo n-grama no aparece en train.
- **% visto ≥ 5 veces**: ocurrencias cuyo n-grama se vio lo suficiente para estimar su probabilidad (cobertura fiable).
- **Pureza**: % de las apariciones en train de ese n-grama que caen en su clase más frecuente (azar = 16.7 %). Solo cuenta los vistos ≥ 5 veces, porque un n-grama visto una vez siempre parece "puro".

In [ ]:
# Compromiso de cada orden n: a mayor n, el n-grama es más específico de una clase (pureza)
# pero es más probable no haberlo visto nunca en train (dispersión). Partición de ajuste de validation.split.
Xtr, ytr, Xva, _ = validation.split(X, y, 20261108)
k = len(CLASES)
MIN_VECES = 5  # un n-grama visto 1-2 veces siempre parece "puro"; solo cuentan los vistos al menos 5 veces
EN_MODELO = {("char", 4), ("char", 5), ("char", 6), ("word", 1), ("word", 3)}
idx_clase = {c: i for i, c in enumerate(CLASES)}


def eventos(texto, unidad, n):
    if unidad == "char":
        t = texto.casefold()
        return [t[i : i + n] for i in range(len(t) - n + 1)]
    toks = TOKEN_RE.findall(texto.casefold())
    return [tuple(toks[i : i + n]) for i in range(len(toks) - n + 1)]


filas = []
for unidad, ordenes in (("caracteres", range(2, 9)), ("palabras", range(1, 4))):
    u = "char" if unidad == "caracteres" else "word"
    for n in ordenes:
        indice, planos = {}, []
        for t, c in zip(Xtr, ytr):
            ci = idx_clase[c]
            planos.extend(indice.setdefault(g, len(indice)) * k + ci for g in eventos(t, u, n))
        conteos = np.bincount(np.asarray(planos, dtype=np.int64), minlength=len(indice) * k).reshape(-1, k)
        total_g = conteos.sum(axis=1)
        pureza_g = conteos.max(axis=1) / total_g
        ids = np.array([indice.get(g, -1) for t in Xva for g in eventos(t, u, n)])
        vistos = ids >= 0
        fiables = np.zeros(len(ids), dtype=bool)
        fiables[vistos] = total_g[ids[vistos]] >= MIN_VECES
        filas.append(
            {
                "unidad": unidad,
                "n": n,
                "% no visto en train": 100 * (~vistos).mean(),
                f"% visto ≥{MIN_VECES} veces": 100 * fiables.mean(),
                f"pureza de los vistos ≥{MIN_VECES} (%)": 100 * pureza_g[ids[fiables]].mean(),
                "en el modelo": (u, n) in EN_MODELO,
            }
        )
        del indice, planos, conteos

dispersion = pd.DataFrame(filas)
dispersion.round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (unidad, d) in zip(axes, dispersion.groupby("unidad", sort=False)):
    ax.plot(d["n"], d["% no visto en train"], "o-", color="#C44E52", label="% no visto en train (dispersión)")
    ax.plot(
        d["n"],
        d[f"% visto ≥{MIN_VECES} veces"],
        "^-",
        color="#8172B2",
        label=f"% visto ≥{MIN_VECES} veces (cobertura fiable)",
    )
    ax.plot(
        d["n"],
        d[f"pureza de los vistos ≥{MIN_VECES} (%)"],
        "s-",
        color="#4C72B0",
        label="pureza (clase más frecuente)",
    )
    ax.axhline(100 / len(CLASES), color="gray", ls=":", lw=1, label="pureza al azar (16.7 %)")
    for _, r in d[d["en el modelo"]].iterrows():
        ax.axvspan(r["n"] - 0.3, r["n"] + 0.3, color="#55A868", alpha=0.15, lw=0)
    ax.set_title(f"N-gramas de {unidad}")
    ax.set_xlabel("orden n")
    ax.set_xticks(d["n"])
axes[0].set_ylabel("%")
axes[1].legend(frameon=False, fontsize=8, loc="center left")
fig.suptitle(
    "Especificidad vs. dispersión por orden (franja verde = órdenes usados en el modelo)", fontsize=10
)
guardar(fig, "dispersion_ngramas")
plt.show()

## 6. Conclusiones

Cada punto resume un hallazgo con el número que lo respalda; están redactados para pasarlos al informe.

**Datos y calidad**

1. **Clases balanceadas.** Cada rango tiene entre 16.0 % y 17.0 % del train, así que el F1 macro y la accuracy casi coinciden y no hace falta reponderar clases.
2. **El texto viene normalizado a ASCII.** Ningún tweet tiene tildes, `ñ`, `¡` ni emojis (`año` aparece como `ano`, `mañana` como `manana`); el único carácter no ASCII es el `¿` de 51 celdas `#¿NOMBRE?`, un error de Excel en la generación del dataset. Por eso no se puede analizar ortografía con tildes ni emojis, y `año`/`ano` son indistinguibles.
3. **Quedan artefactos de origen.** 2 199 tweets (1.3 %) tienen entidades HTML sin decodificar (`&lt;3` es un corazón `<3`), más frecuentes en 13-17 (2.6 %). 4 211 tweets (2.5 %) llegan al tope de 140 caracteres y están truncados, sobre todo en las clases mayores (3.5–3.8 %).

**Duplicados y plantillas**

4. **Copias exactas: ruido pequeño.** Solo el 1.2 % del train se repite literalmente, pero 643 tweets están en textos repetidos con más de una etiqueta: es error irreducible. La validación ya agrupa por texto, así que estas copias no inflan la métrica.
5. **Plantillas: 10 % del corpus, pero poca señal.** Al reemplazar URLs, menciones, hashtags y números, el 10.2 % del train y el 10.0 % del test comparten plantilla con algún tweet de train. Las más grandes casi no informan la edad: predecir la clase mayoritaria de la plantilla acierta 24 % en `URL` sola y 21 % en `RT @U: URL` (azar 16.7 %). Solo las plantillas casi exclusivas de una clase (≥ 90 %, generalmente bots o una misma cuenta, como `#MTVHottestJustinBieber` o Favstar) aciertan 96 %, y cubren ~1.7 % de los tweets.
6. **El 5.9 % de los tweets no tiene texto propio** (solo enlaces, menciones, hashtags o `RT`). Para ellos la edad solo puede inferirse de a quién mencionan o qué comparten.

**Longitud y estilo**

7. **La longitud crece con la edad.** Mediana de 66 caracteres en 13-17 y 18-24, 84 en 25-34, 103 en 35-49 y 131 en 65+. La longitud separa 18-24 de 25-34, pero no 13-17 de 18-24.
8. **Los jóvenes escriben más expresivo.** Alargamientos (`siii`) en el 6.9 % de 13-17 frente a 1.6 % en 65+; risas y emoticonos superan el 3–5 % hasta 25-34 y caen por debajo del 1 % desde 50-64.
9. **Los mayores comparten más.** URLs (33.5 % en 13-17 → 50.7 % en 65+), números (45.5 % → 60.7 %), y en 65+ dominan los retweets (53.4 %) y las menciones dentro del texto (61.5 %).
10. **25-34 frente a 18-24: más enlaces y puntuación.** Las mayores diferencias son URLs (38.3 % vs 30.1 %) y números (48.6 % vs 41.6 %); además 25-34 usa más `!` (15.0 % vs 11.5 %) y `...` (10.0 % vs 6.7 %), responde más con mención al inicio (20.3 % vs 17.1 %) y usa menos emoticonos (3.4 % vs 5.2 %). Las respuestas crecen hasta 50-64 (27.6 %) y caen en 65+ (16.6 %), donde el RT reemplaza a la respuesta.
11. **Las mayúsculas tienen forma de U.** El % de letras en mayúscula baja de 15.4 % (13-17) a 12.3 % (25-34) y vuelve a 15.3 % (65+); los tweets "gritados" siguen el mismo patrón (2.4 % → 1.0 % → 2.6 %).

**Menciones y vocabulario**

12. **Las menciones son casi una firma de comunidad.** De los usuarios mencionados 5 o más veces, el 47.9 % aparece en una sola clase al menos el 90 % de las veces, y el 75.7 % de las menciones de test ya aparece en train. Esto explica por qué el canal de menciones es el que más aporta a la mezcla: la señal es en buena parte social, no lingüística.
13. **Cada clase tiene temas reconocibles, pero poco frecuentes.** Sin stopwords: fandom en 13-17 (`justin`, `bieber`, `vote`, `sigueme`), lenguaje afectivo en 18-24 (`quiero`, `amo`, `contigo`, `necesito`) y mensajes de apps en portugués (`seu perfil`, `pessoas`), publicaciones automáticas en 35-49 (`post`, `disponible`, `stories`), conflicto colombiano en 50-64 (`paz`, `farc`, `narco`, `secuestrados`) y política venezolana en 65+ (`maduro`, `revolucion`, `patria`). Sin embargo, las 20 palabras más características de cada clase aparecen en solo el 6.5–14.9 % de sus tweets: la señal de edad está repartida en el estilo y el vocabulario general, no en unas pocas palabras clave.
14. **18-24 vs 25-34: registro personal frente a informativo.** 18-24 sobreusa pronombres (`me` z = 20, `te` 20, `yo` 11); 25-34 sobreusa artículos y preposiciones (`de` 18, `en` 14, `del` 12). Las palabras de etapa de vida van en la misma dirección: `novio`/`novia`, `mama`, `colegio` y `examen` son más jóvenes; `trabajo`, `oficina` y `jefe`, más adultas.
15. **25-34 no tiene vocabulario propio.** Su palabra más característica frente al resto tiene z = 11, contra 30–49 en las otras clases: es una clase de transición, mezcla de sus vecinas. También es la clase cuyas top-20 cubren menos tweets (6.5 %). Esto es coherente con que sea la de menor F1 (0.477).
16. **La distancia léxica es ordinal.** El número de palabras que separan dos clases crece con la diferencia de edad: 235–455 entre clases vecinas y 1 249 entre 13-17 y 65+. 18-24 está más cerca de 13-17 (235) que de 25-34 (285). Por eso los errores se concentran entre clases contiguas.

**N-gramas**

17. **Los n-gramas de caracteres capturan el estilo.** Las risas son lo más distintivo de 18-24 (`jaja`, z = 60, el valor más alto de todo el análisis), junto con alargamientos (`oooo`); 13-17 se distingue por `follow` y el corazón `&lt;3`, 25-34 por los puntos suspensivos y 65+ por la terminación `-cion` (registro formal). Esto explica por qué `char5` y `char6` son los mejores canales individuales (F1 ≈ 0.54).
18. **Los emoticonos y la puntuación repetida solo existen como pares.** El tokenizador separa `:)` en `:` + `)`, de modo que la sonrisa solo aparece como bigrama (`: )` es el bigrama más característico de 18-24); lo mismo pasa con `! !` y `. .` en 25-34. Algunos patrones son bots: `seu perfil foi visto por pessoas` (18-24) o los check-ins de Foursquare `(@ ` (25-34).
19. **Los órdenes 4–6 de caracteres son un buen compromiso.** Al subir el orden, la pureza crece (21 % → 40 % de 2 a 8 caracteres), pero cae la cobertura fiable: con 4–6 caracteres el 77–94 % de los n-gramas de validación se vio al menos 5 veces en train; con 8, solo el 54 %. En palabras, los trigramas son los más específicos (48 %) pero solo el 15.5 % se vio 5 veces o más y el 68 % nunca se vio: el LM de palabras de orden 3 depende sobre todo del suavizado, lo que explica su F1 individual bajo (0.37) aunque aporte a la mezcla.

### Qué no capturan los modelos actuales

- Todos los canales aplican `casefold`, así que se pierde el patrón en U de las mayúsculas (punto 11).
- La longitud no se modela como tal: los LMs secuenciales dividen el puntaje por el número de tokens y los multinomiales solo la usan como escala de confianza (punto 7).
- `word_seq3` reduce URLs, menciones y hashtags a `URL`, `@USER` y `#TAG`; esa información solo llega por los canales específicos (puntos 9 y 12).
- Las entidades HTML se tokenizan como `&`, `lt`, `;`, así que el corazón `<3` y `&lt;3` son eventos distintos (punto 3).
- Los canales de palabras ven los emoticonos partidos (`:` y `)` como tokens separados); solo los n-gramas de caracteres y los bigramas los capturan completos (punto 18).

### Observaciones para revisar después

No se prueban aquí; quedan anotadas para `02_experimentos`.

- La **longitud** separa 18-24 de 25-34 (mediana 66 vs 84) y ningún canal la modela de forma explícita.
- Las **mayúsculas** se pierden por el `casefold` y tienen un patrón claro por edad.
- **Filtrar de train las plantillas ruidosas** (pureza ≤ 50 %, n ≥ 5, calculada solo con el train de cada semilla) y comparar en las dos semillas; si mejora, revisar si la ganancia viene de los tweets sin plantilla o del cambio de priors (punto 5).
- El **1.2 % del test es copia exacta** de un tweet de train (10 % comparte plantilla). Usar la etiqueta de train para esos casos hay que **consultarlo con el profesor** antes.
- La **submission actual está desbalanceada**: predice 21 770 tweets como 18-24 y 15 434 como 35-49, cuando el test debería tener ~18 800 por clase. Sobre-predecir una clase y sub-predecir otra resta F1 macro.